# SkyEye — 天气分类
## EfficientNet-B5 → 知识蒸馏 → B0 → 结构化剪枝 → ONNX 导出

按顺序执行以下 Cell：

In [ ]:
# Cell 8 (可选): 单张图片推理测试
from inference.infer import predict_image
result = predict_image('test_image.jpg')  # 替换为实际图片路径
print(f'Prediction: {result["prediction"]} (Confidence: {result["confidence"]:.4f})')
for name, prob in result['top_k']:
    print(f'  {name}: {prob:.4f}')

In [ ]:
# Cell 7: ONNX 导出 + 推理测速
from inference.export_onnx import export_to_onnx, benchmark_inference
onnx_path = export_to_onnx('results/student_pruned_final.pth')
benchmark_inference('results/student_pruned_final.pth', onnx_path)

In [ ]:
# Cell 6: 结构化剪枝 + 微调
from training.prune_finetune import prune_and_finetune
pruned_model = prune_and_finetune()

In [ ]:
# Cell 5: 知识蒸馏 (B5 → B0)
from training.distill_student import run_distillation
student = run_distillation()

In [ ]:
# Cell 4: 训练教师模型 (EfficientNet-B5)
from training.train_teacher import train_teacher
teacher = train_teacher()

In [ ]:
# Cell 3: 数据准备（复制数据集到可写目录 + 验证）
from data.dataset import prepare_data, create_dataloaders
data_root = prepare_data()
print(f'Data ready at: {data_root}')

# 快速验证数据加载
train_loader, val_loader, class_counts = create_dataloaders()
print(f'Train batches: {len(train_loader)}, Val batches: {len(val_loader)}')

In [ ]:
# Cell 2: 检查环境和配置
from config import CONFIG
import torch
print(f'Device: {CONFIG["device"]}')
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'Classes: {CONFIG["class_names"]}')
print(f'Teacher: {CONFIG["teacher_model"]}, Student: {CONFIG["student_model"]}')

In [ ]:
# Cell 1: 安装依赖
!pip install torch torchvision timm onnx onnxruntime-gpu tqdm scikit-learn